# Knowledge Base Assistant (Google Gemini + RAG)
نسخة نوتبوك من ملف `all_in_one.py` — نفس الكود بالضبط، مقسّم لخلايا حسب الأقسام الأربعة (Config, Loaders, RAG Engine, Main).

⚠️ **للتشغيل والمراجعة هون بكولاب. أما تسليم المشروع لـ GitHub، لازم يضل بالهيكلية المقسّمة (main.py + src/).**

In [1]:
!pip install -q -U langchain langchain-google-genai langchain-community langchain-text-splitters langchain-chroma chromadb pypdf docx2txt pandas python-dotenv

## إعداد مفتاح Gemini
جيب مفتاح مجاني من `https://aistudio.google.com/apikey`، وحطه بالـ Secrets بكولاب باسم `GOOGLE_API_KEY` (نفس طريقة GROQ_API_KEY يلي سويناها قبل).

In [2]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("API key loaded")

API key loaded


## إنشاء مجلد Knowledge Base
ارفع ملفاتك (PDF, DOCX, TXT, MD, CSV) داخل هالمجلد قبل ما تكمل.

In [3]:
import os
os.makedirs("knowledge_base", exist_ok=True)
print("Upload your files into the 'knowledge_base' folder (use the Files panel on the left), then continue.")

Upload your files into the 'knowledge_base' folder (use the Files panel on the left), then continue.


---
## SECTION 1 — CONFIG

In [4]:
import sys
import shutil
from typing import List, Tuple

from langchain_core.documents import Document
from langchain_community.document_loaders import (
    PyPDFLoader,
    Docx2txtLoader,
    TextLoader,
    CSVLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

# Updated KNOWLEDGE_BASE_DIR to point to the actual documents after extraction
KNOWLEDGE_BASE_DIR = os.environ.get("KNOWLEDGE_BASE_DIR", "knowledge_base/kb-assistant/knowledge_base")
VECTOR_STORE_DIR = os.environ.get("VECTOR_STORE_DIR", ".vector_store")

LLM_MODEL_NAME = os.environ.get("LLM_MODEL_NAME", "gemini-2.5-flash")
EMBEDDING_MODEL_NAME = os.environ.get("EMBEDDING_MODEL_NAME", "models/gemini-embedding-001")

CHUNK_SIZE = int(os.environ.get("CHUNK_SIZE", "1000"))
CHUNK_OVERLAP = int(os.environ.get("CHUNK_OVERLAP", "150"))
RETRIEVAL_TOP_K = int(os.environ.get("RETRIEVAL_TOP_K", "4"))

SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".txt", ".md", ".csv"}


def validate_config():
    """Raises a clear error early if required configuration is missing."""
    if not GOOGLE_API_KEY:
        raise ValueError(
            "GOOGLE_API_KEY is not set. Add it via Colab Secrets before continuing."
        )

print("Config loaded")


/tmp/ipykernel_29404/2386095189.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


Config loaded


---
## SECTION 2 — LOADERS
قراءة كل أنواع الملفات المدعومة من مجلد `knowledge_base`.

In [5]:
LOADER_MAP = {
    ".pdf": PyPDFLoader,
    ".docx": Docx2txtLoader,
    ".txt": TextLoader,
    ".md": TextLoader,
    ".csv": CSVLoader,
}


def discover_files(directory: str = KNOWLEDGE_BASE_DIR) -> List[str]:
    """Returns a list of full file paths for every supported file
    found directly inside `directory` (not sub-folders)."""
    if not os.path.isdir(directory):
        return []

    found = []
    for filename in os.listdir(directory):
        full_path = os.path.join(directory, filename)
        if not os.path.isfile(full_path):
            continue
        _, ext = os.path.splitext(filename)
        if ext.lower() in SUPPORTED_EXTENSIONS:
            found.append(full_path)
    return found


def load_single_file(file_path: str) -> List[Document]:
    """Loads one file into a list of LangChain Document objects,
    using whichever loader class matches its extension."""
    _, ext = os.path.splitext(file_path)
    ext = ext.lower()

    loader_class = LOADER_MAP.get(ext)
    if loader_class is None:
        raise ValueError(f"Unsupported file type: {ext}")

    if loader_class is TextLoader:
        loader = loader_class(file_path, encoding="utf-8")
    else:
        loader = loader_class(file_path)

    documents = loader.load()

    for doc in documents:
        doc.metadata["source_file"] = os.path.basename(file_path)

    return documents


def load_all_documents(directory: str = KNOWLEDGE_BASE_DIR) -> List[Document]:
    """Discovers and loads every supported file in the Knowledge Base directory.
    Files that fail to load are skipped with a warning instead of crashing
    the whole ingestion process."""
    all_documents: List[Document] = []
    file_paths = discover_files(directory)

    if not file_paths:
        print(f"[loaders] No supported files found in '{directory}'.")
        return all_documents

    for path in file_paths:
        try:
            docs = load_single_file(path)
            all_documents.extend(docs)
            print(f"[loaders] Loaded {len(docs)} section(s) from {os.path.basename(path)}")
        except Exception as e:
            print(f"[loaders] Skipped '{os.path.basename(path)}' due to an error: {e}")

    return all_documents

print("Loaders defined")

Loaders defined


---
## SECTION 3 — RAG ENGINE
التقطيع، الفهرسة (embeddings + Chroma)، والاسترجاع + الجواب المقيّد بالسياق.

In [6]:
GROUNDED_PROMPT_TEMPLATE = """You are a Knowledge Base Assistant. Answer the
question using ONLY the context provided below, which was retrieved from the
user's own documents.

Rules:
- If the answer is fully contained in the context, answer it clearly and cite
  which source file(s) it came from.
- If the context does NOT contain enough information to answer, respond
  exactly with: "I don't have enough information in the Knowledge Base to
  answer that." Do not guess or use outside knowledge.

Context:
{context}

Question: {question}

Answer:"""


import chromadb # Added import for chromadb client
import uuid # Added for generating unique IDs
import time # Added for small delays

def split_documents(documents: List[Document]) -> List[Document]:
    """Splits loaded documents into smaller overlapping chunks so the
    embedding model and retrieval step both work more accurately."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    return splitter.split_documents(documents)


def build_vector_store(chunks: List[Document]) -> Chroma:
    """Embeds every chunk and stores it in a fresh Chroma vector database
    on disk, replacing any previous index."""
    embeddings = GoogleGenerativeAIEmbeddings(
        model=EMBEDDING_MODEL_NAME,
        google_api_key=GOOGLE_API_KEY,
    )

    if os.path.isdir(VECTOR_STORE_DIR):
        shutil.rmtree(VECTOR_STORE_DIR)
        time.sleep(0.1) # Add a small delay to ensure file handles are released
    os.makedirs(VECTOR_STORE_DIR, exist_ok=True)

    chroma_client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)
    collection_name = "knowledge_base_collection"

    # Explicitly ensure the collection is new or clean
    try:
        # Attempt to delete the collection first to ensure a fresh start
        chroma_client.delete_collection(name=collection_name)
        print(f"[build_vector_store] Deleted existing collection '{collection_name}'.")
    except Exception as e:
        # Collection might not exist, or an error occurred during deletion, which is fine
        print(f"[build_vector_store] Warning: Could not delete old collection '{collection_name}': {e}")
        pass # Proceed with creation if deletion fails or collection doesn't exist

    # Create a fresh collection
    collection = chroma_client.create_collection(name=collection_name)
    print(f"[build_vector_store] Created new collection '{collection_name}'.")

    # Prepare data for direct upsert to chromadb collection
    # Generate unique IDs for each chunk. UUIDs are good for uniqueness.
    ids = [str(uuid.uuid4()) for _ in chunks]
    documents_content = [chunk.page_content for chunk in chunks]
    metadatas = [chunk.metadata for chunk in chunks]

    # Get embeddings for all chunks. This calls the Gemini embeddings API once for all.
    embeddings_list = embeddings.embed_documents(documents_content)

    # Directly add to the chromadb collection
    collection.add(
        documents=documents_content,
        metadatas=metadatas,
        embeddings=embeddings_list,
        ids=ids
    )
    print(f"[build_vector_store] Added {len(chunks)} chunks to collection.")

    # Finally, create the langchain Chroma object from this existing collection
    # This object then interacts with the `chroma_client` and `collection_name`
    vector_store = Chroma(
        client=chroma_client,
        collection_name=collection_name,
        embedding_function=embeddings,
    )
    return vector_store


def load_vector_store() -> Chroma:
    """Loads a previously built vector store from disk, without
    re-embedding anything (fast startup for repeated runs)."""
    embeddings = GoogleGenerativeAIEmbeddings(
        model=EMBEDDING_MODEL_NAME,
        google_api_key=GOOGLE_API_KEY,
    )
    # Explicitly initialize ChromaDB client to load the existing collection
    chroma_client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)
    return Chroma(
        client=chroma_client,
        collection_name="knowledge_base_collection", # Must match the name used in build_vector_store
        embedding_function=embeddings,
    )


def get_llm() -> ChatGoogleGenerativeAI:
    """Creates the Gemini chat model instance used to generate answers."""
    return ChatGoogleGenerativeAI(
        model=LLM_MODEL_NAME,
        google_api_key=GOOGLE_API_KEY,
        temperature=0,
    )


def answer_question(vector_store: Chroma, llm: ChatGoogleGenerativeAI, question: str) -> Tuple[str, List[str]]:
    """Retrieves the most relevant chunks for `question`, builds a grounded
    prompt, and asks Gemini to answer. Returns the answer text plus the list
    of source filenames used."""
    relevant_chunks = vector_store.similarity_search(question, k=RETRIEVAL_TOP_K)

    if not relevant_chunks:
        return "I don't have enough information in the Knowledge Base to answer that.", []

    context = "\n\n---\n\n".join(chunk.page_content for chunk in relevant_chunks)
    sources = sorted({chunk.metadata.get("source_file", "unknown") for chunk in relevant_chunks})

    prompt = GROUNDED_PROMPT_TEMPLATE.format(context=context, question=question)
    response = llm.invoke(prompt)

    return response.content, sources

print("RAG engine defined")

RAG engine defined


---
## SECTION 4 — MAIN (بناء الفهرس ثم المحادثة)

In [7]:
def build_index():
    """Runs the full ingestion pipeline: load files -> split -> embed -> store."""
    print(f"\n[main] Scanning '{KNOWLEDGE_BASE_DIR}' for documents...")
    documents = load_all_documents()

    if not documents:
        print(
            "[main] No documents were loaded. Add PDF, DOCX, TXT, MD, or CSV "
            f"files to the '{KNOWLEDGE_BASE_DIR}' folder and run again."
        )
        return None

    print(f"[main] Splitting {len(documents)} document section(s) into chunks...")
    chunks = split_documents(documents)
    print(f"[main] Created {len(chunks)} chunk(s). Building the vector index (this calls the Gemini embeddings API)...")

    vector_store = build_vector_store(chunks)
    print("[main] Vector index built successfully.\n")
    return vector_store


def chat_loop(vector_store):
    """Interactive question-answer loop (works fine in a notebook cell too)."""
    llm = get_llm()
    print("Knowledge Base Assistant is ready. Type your question, or 'exit' to quit.\n")

    while True:
        question = input("You: ").strip()

        if not question:
            continue
        if question.lower() in {"exit", "quit"}:
            print("Goodbye.")
            break

        try:
            answer, sources = answer_question(vector_store, llm, question)
        except Exception as e:
            print(f"Assistant: Sorry, something went wrong while answering: {e}\n")
            continue

        print(f"\nAssistant: {answer}")
        if sources:
            print(f"(Sources: {', '.join(sources)})")
        print()

print("Main functions defined")

Main functions defined


## التشغيل الفعلي
شغّل الخلية الجاية بعد ما ترفع ملفاتك لمجلد `knowledge_base` — رح تبني الفهرس أول مرة وتبلش المحادثة مباشرة.

In [8]:
import os
if os.path.exists('.vector_store'):
    !rm -rf .vector_store
    print("Removed existing .vector_store directory.")
else:
    print("'.vector_store' directory not found, no action needed.")

Removed existing .vector_store directory.


يمكنك التحقق من النماذج المتاحة حاليًا لميزات `embedContent` باستخدام الكود التالي:

In [9]:
import google.generativeai as genai

for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


بعد تشغيل الخلية أعلاه، ستظهر لك قائمة بنماذج التضمين المدعومة. اختر واحدًا (عادةً `models/embedding-001` قد يكون متاحًا أحيانًا، ولكن إذا لم يكن، اختر بديلاً مثل `models/text-embedding-004`)، ثم **قم بتعديل خلية الإعدادات `cElt-esS03gp`** لتحديث قيمة `EMBEDDING_MODEL_NAME`.

بعد تعديل الخلية `cElt-esS03gp` وتشغيلها، أعد تشغيل الخلية `ofagvwuQ03gq` مرة أخرى.

In [10]:
import zipfile
import os

zip_file_path = os.path.join("knowledge_base", "kb-assistant.zip")
extract_dir = "knowledge_base"

if os.path.exists(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"Extracted files from {zip_file_path} to {extract_dir}/")
    # Optionally, remove the zip file after extraction
    # os.remove(zip_file_path)
else:
    print(f"Zip file not found at {zip_file_path}")


Zip file not found at knowledge_base/kb-assistant.zip


In [11]:
import os

print(f"Contents of '{KNOWLEDGE_BASE_DIR}':")
for root, dirs, files in os.walk(KNOWLEDGE_BASE_DIR):
    level = root.replace(KNOWLEDGE_BASE_DIR, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')


Contents of 'knowledge_base/kb-assistant/knowledge_base':
knowledge_base/
    .gitkeep
    file 1.pdf


In [ ]:
import os

# 1. Check where we are
print("Current directory:", os.getcwd())

# 2. If the project is on Google Drive, copy it locally first
if "/content/drive" in os.getcwd():
    get_ipython().system('cp -r "/content/drive/MyDrive/kb-assistant" "/content/kb-assistant"')
    get_ipython().magic('cd /content/kb-assistant')

# 3. Find and remove any old Chroma database folders
get_ipython().system('find . -iname "*chroma*" -maxdepth 3')
get_ipython().system('rm -rf chroma_db knowledge_base/chroma_db')

# 4. Rebuild the index
validate_config()
vector_store = build_index()
if vector_store is not None:
    chat_loop(vector_store)

Current directory: /content
find: warning: you have specified the global option -maxdepth after the argument -iname, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.

[main] Scanning 'knowledge_base/kb-assistant/knowledge_base' for documents...
[loaders] Loaded 6 section(s) from file 1.pdf
[main] Splitting 6 document section(s) into chunks...
[main] Created 13 chunk(s). Building the vector index (this calls the Gemini embeddings API)...
[build_vector_store] Warning: Could not delete old collection 'knowledge_base_collection': Collection [knowledge_base_collection] does not exist
[build_vector_store] Created new collection 'knowledge_base_collection'.
[build_vector_store] Added 13 chunks to collection.
[main] Vector index built successfully.

Knowledge Base Assistant is ready. Type your question, or 'exit' to quit.

You: what's the data mining 

Assistant: 